In [12]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-z7io637r/polara_b6c73dbe702c4affab57e76ea918ac6d
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-z7io637r/polara_b6c73dbe702c4affab57e76ea918ac6d
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]


In [13]:
from tqdm.auto import tqdm

from polara import get_movielens_data


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch

from lib.dataprep import (transform_indices, verify_time_split, reindex_data, leave_last_out, tvt_data_split, encode_entity_features)


In [17]:
data = get_movielens_data(include_time=True, get_genres=False, local_file='ml-20m.zip')
items_metadata_ = pd.read_csv('https://raw.githubusercontent.com/evfro/recsys19_hybridsvd/master/data/meta_info_ml1m.csv', sep=';')

In [18]:
training_, validation_, testset_ = tvt_data_split(data, data['timestamp'].quantile(0.9), data['timestamp'].quantile(0.95))


len(training_), len(validation_), len(testset_)

(18000236, 1000013, 1000014)

In [21]:
training, data_index = transform_indices(training_, users='userid', items='movieid')

validation_ = reindex_data(validation_, data_index, entities=['users', 'items'], filter_invalid=True)
validation, validation_holdout = leave_last_out(validation_)
testset_ = reindex_data(testset_, data_index, entities=['users', 'items'], filter_invalid=True)
testset, testset_holdout = leave_last_out(testset_)

items_metadata = reindex_data(items_metadata_, data_index, entities=['items'], filter_invalid=True)


verify_time_split(training, validation)
verify_time_split(training, testset)
verify_time_split(validation, testset)

In [ ]:
data_description = dict(
    users = data_index['users'].name,
    items = data_index['items'].name,
    feedback = 'rating',
    n_users = len(data_index['users']),
    n_items = len(data_index['items']),
    val_users = np.sort(validation_holdout[data_index['users'].name].values),
    test_users = np.sort(testset_holdout[data_index['users'].name].values),
    val_targets = validation_holdout.groupby(data_index['users'].name)[data_index['items'].name].apply(list).to_dict(),
    test_targets = testset_holdout.groupby(data_index['users'].name)[data_index['items'].name].apply(list).to_dict()
)